Statquest
Stanford CME295 Transformers & LLMs 
Attention is all you need (sinusoidal positional encodings)
ALiBi and RoPE scaling
self head attention
encoder-decoder attention
only encoder attn
only decoder attn
tf model
MHA, GQA, MQA, and Multi-Head Latent Attention

In [ ]:
'''https://youtu.be/LBsyiaEki_8
learn with jaay transformers

https://youtu.be/k86eUj4hgdc
Positional Encodings and Group Theory | 3Blue1Brown and Alok Puranik

Ali et al., Tokenizer Choice for LLM Training: Negligible or Crucial? (2023)
Im going with 33k vocab size as higher vocab sizes (higher than 55k) require more compute for 
english-only data, and our data is engligh only.
BPE-SP-33 is to be used.'''

In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool

In [ ]:

HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"


# Destination HF repo for uploading your trained tokenizer (e.g. "your-username/legal-bpe-tokenizer")
# Set your HF username or desired repo ID below:
HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")


TRAIN_FILES = ["Phase_1_Pretraining/Train/train_supreme_court.json", "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json", "Phase_1_Pretraining/Train/train_tribunals.json"]

TEST_FILES = ["Phase_1_Pretraining/Test/Phase1- Test.json", "Phase_1_Pretraining/Test/test.json"]

TOKEN_BUDGETS = {"train_supreme_court.json": 10_000_000, "train_acts.json": 5_000_000,
    "train_high_courts.json": 5_000_000, "train_tribunals.json": 5_000_000,}


# TRAIN_TOKENS = 200_000_000
# VAL_TOKENS = 5_000_000

# print("Training token budget:", TRAIN_TOKENS)
# print("Validation token budget:", VAL_TOKENS)


TRAIN_BLOCK_SIZE = 2048
MODEL_CONTEXT_LENGTH = 2048

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
EPOCHS = 1
TARGET_WORDS = 128
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.95
TOP_K = 30
REPETITION_PENALTY = 1.2


KNOWN_COUNTS = {"train_supreme_court.json": 55549, "train_acts.json": 18826,
                "train_high_courts.json": 2536738, "train_tribunals.json": 869132}

CTX = 2048
HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase1"

OUTPUT_DIR = "/kaggle/working/cs781_phase1" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")

N_EMBD = 560
N_LAYER = 8
N_HEAD = 8


print("\nConfiguration loaded")
print(f"Target Token Splits: {TOKEN_BUDGETS}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Tokenizer repository: {HF_TOK_REPO}")
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Context length: {MODEL_CONTEXT_LENGTH}")
print(f"Model: {N_LAYER} layers × {N_HEAD} heads × {N_EMBD} embedding")

In [ ]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 1 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_path = None
    for tf in TEST_FILES:
        try:
            test_path = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            print(f"Downloaded Test File: {tf}")
            break
        except Exception:
            continue
            
    return local_train_paths, test_path

In [ ]:
train_paths, test_path = download_data()

# inspect_json_files(train_paths, n=250)

In [ ]:
'''!pip install ijson
import ijson
import os


def inspect_first_entries(file_paths, n=2):

    for path in file_paths:

        print("FILE:", path)

        size_gb = os.path.getsize(path) / (1024**3)
        print(f"File size: {size_gb:.2f} GB")

        with open(path, "rb") as f:

            entries = ijson.items(f, "item")

            for i, entry in enumerate(entries):

                if i >= n:
                    break

                print(f"\n--- Entry {i+1} ---")
                print(entry)'''

In [ ]:
'''inspect_first_entries(train_paths, n=2)'''

In [ ]:
'''import ijson
import os

def count_entries(file_paths):

    for path in file_paths:

        print("=" * 100)
        print("FILE:", path)

        count = 0

        with open(path, "rb") as f:
            entries = ijson.items(f, "item")

            for _ in entries:
                count += 1

        print("Total entries:", count)

count_entries(train_paths)'''


'''
====================================================================================================
FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
Total entries: 55549
====================================================================================================
FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
Total entries: 18826
====================================================================================================
FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
Total entries: 2536738
====================================================================================================
FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json
Total entries: 869132
'''

The output to the above cell is: 

FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
Total entries: 55549

FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
Total entries: 18826

FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
Total entries: 2536738

FILE: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json
Total entries: 869132


--- Entry 1 --- {'title': 'His Holiness Sri Vishwothama Thirtha ... vs The State Of Mysore', 'date': '1961-12-18', 'court': 'Supreme Court of India', 'citations': 'AIR 1966 SUPREME COURT 1882, 1962 2 SCJ 131', 'bench': '', 'headnote': 'The shrine in suit which was originally founded by Shri Madhavacharya, a Hindu saint was managed for a long time in rotation by the heads of eight Mutts which were also founded by the said saint for eight of his disciples. Large number of pilgrims from all over the country visited this shrine without any restraint, took part in the worship there, made offerings to the deity and received the prasad. The institution was managed with the monetary aid received from the State and contributions raised by the said heads of the eight Mutts from devotees resident in different parts of the country. The question arising for decision was whether the shrine in question was a "temple" within the meaning of s. 2(1) of the Madras Temple Entry Authorization Act, 1947. ^ Held, that in the absence of good evidence that a temple was a private one, the mere fact that it was visited by a large number of persons among the Hindu Public without any restraint for a number of years, was good evidence of the fact that the temple had been dedicated to the Hindu Public and was for its benefit. Vibhudapriya v. Lakshmindra, (1927) L. R. 54 I. A. 228, referred to. Babu Bhagwan Din v. Gir Har Saroop, (1939) L. R. 67 I. A. 1, and Sri Venkataramana Devarau v. State of Mysore, [1958] S.C.R. 895, followed. In the instant case the finding that the Hindu Public had a right to worship in the temple was sufficient to make the institution a "temple" within the definition of that term in the Act even if the temple be appurtenant to a Mutt.', 'text': 'PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs.\n\nRESPONDENT: THE STATE OF MYSORE\n\nDATE OF JUDGMENT: 18/12/1961\n\nACT: Temple-Large number of pilgrims visiting and worshiping-If private or public-Madras Temple Entry Authorisation Act, 1947(Mad. V. of 1947), s. 2(1).\n\nHEADNOTE: The shrine in suit which was originally founded by Shri Madhavacharya, a Hindu saint was managed for a long time in rotation by the heads of eight Mutts which were also founded by the said saint for eight of his disciples. Large number of pilgrims from all over the country visited this shrine without any restraint, took part in the worship there, made offerings to the deity and received the prasad. The institution was managed with the monetary aid received from the State and contributions raised by the said heads of the eight Mutts from devotees resident in different parts of the country. The question arising for decision was whether the shrine in question was a "temple" within the meaning of s. 2(1) of the Madras Temple Entry Authorization Act, 1947. Held, that in the absence of good evidence that a temple was a private one, the mere fact that it was visited by a large number of persons among the Hindu Public without any restraint for a number of years, was good evidence of the fact that the temple had been dedicated to the Hindu Public and was for its benefit. Vibhudapriya v. Lakshmindra, (1927) L. R. 54 I. A. 228, referred to. Babu Bhagwan Din v. Gir Har Saroop, (1939) L. R. 67 I. A. 1, and Sri Venkataramana Devarau v. State of Mysore, [1958] S.C.R. 895, followed. In the instant case the finding that the Hindu Public had a right to worship in the temple was sufficient to make the institution a "temple" within the definition of that term in the Act even if the temple be appurtenant to a Mutt.\n\nJUDGMENT:\n\nCIVIL APPELLATE JURISDICTION: Civil Appeal No. 84 of 1959. Appeal from the judgment and decree dated August 17, 1954, of the Madras High Court in A. S. No. 304 of 1951, A. V. Viswanatha Sastri and R. Gopalakrishnan, for the appellants. B. R. L. Iyengar and T. M. Sen, for respondent No. 1,\n\n1961. December 18. The Judgment of the Court was delivered by RAGHUBAR DAYAL, J.-The only question for determination in this appeal on a certificate granted by the Madras High Court is whether what has been described in the plaint as Shri Krishna Mutt in Shivalli Village, in South Kanara District, is a temple as defined in s. 2(1) of the Madras Temple Entry Authorization Act, 1947 (Madras Act V of 1947), hereinafter called the Act. Sub-section (1) of s. 2 of the Act reads: "temple\' means a place, by whatever name known, which is dedicated to, or for the benefit of, or used as of right by, the Hindu community or any section thereof, as a place of public religious worship, and includes subsidiary shrines and mantapams attached to such place;" The Government of Madras, in exercise of the powers conferred on them under s. 6 of the Act, decided on June 3, 1948, that Shri Krishna Mutt was a temple. The plaintiffs, who represent the heads of six of the eight mutts established at Udipi by Shri Madvacharya, instituted the suit for declaration that the Shri Krishna Mutt (not one of the eight mutts) was not such a temple and for the setting aside of the order of the Government of Madras. The heads of the other two mutts were impleaded as defendants Nos. 2 and 3. The State of Madras is defendant No. 1. The State of Madras alone contested the suit and stated that the Shri Krishna Mutt was a temple as defined in the Act. The Trial Court and the High court held in favour of the defendant\'s contention with the result that the plaintiff\'s suit stood dismissed. The plaintiffs have therefore come up in appeal. The case of the plaintiffs-appellants as disclosed in the plaint may first be stated. The great Saint and Savant Shri Madvacharya established the Mutt at Udipi in South Kanara District, for the propagation of the Dwaita system of thought. He himself resided there. He initiated eight disciples and thus originated the Astha Mutts\' (eight mutts) of Udipi. These disciples had to reside, in the mutts and engage themselves in the study and propagation of learning. The way in which the image of Shri Krishna was installed in the Mutt is described thus: "That the Shri Madva Vijaya\', the life history of the great teacher has it, that the Shri Krishna\'s image was acquired from a heap of Gopi Chandan from Dwarka brought by the sailors of a native craft which got stranded near Malpe where the teacher used to go every morning for his ablutions and tapas. Tradition has it that the image is the same that had been worshipped by Rukmini at Dwaraka in pre-historic days. The image was consecrated and installed by the great teacher in his mutt at Udipi and he began to worship it. This according to history and tradition is the origin of what came to be called popularly the Shri Krishna Mutt at Udipi." Shri Madvacharya, who installed the consecrated idol in a shrine of his Mutt, worshipped it during his life-time and ordained that the worship be continued by one or other of his eight disciples. In order to regularise the worship of Shri Krishna and to give equal opportunities to the eight disciples it was arranged that the worship be conducted by each of the disciples by rotation (pariyayam). Worship has continued according to this system for the several centuries that the Shrine has been in existence. The head of each of the eight mutts conducts the worship for a period of two years. During this period he occupies the Gadi or the seat of Shri Madvacharya and is called the Pariyayam Swami. The other seven heads of the mutts, when free from the duty of worshipping the image of Sri Krishna, began to take their abode in the vicinity of the Mutt and thus came into existence the eight independent mutts, though the main Mutt started by Shri Madvacharya, who consecrated and installed the Shri Krishna image, continued as the Shri Krishna Mutt of Udipi. The Pariyayam Swami meets the expenses of the worship and other incidental expenses from the income of his own mutt and the income of the Shri Krishna Mutt. He conducts the worship himself. Shri Krishna Mutt is a shrine attached to the ancient Mutt started by Shri Madvacharya and is not a place of public worship. It has no individual existence apart from the Mutt. The shrine is not a place dedicated to, or for the benefit of, or used as of right by, the Hindu community in general, as a place of public religious Worship, but is intended for the Worship of the eight disciples only. It is however stated, in the Words of the plaint that worship by others could be offered only through Pariyayam Swami, with his consent and as a part of his ordinary worship.\' Defendants Nos. 2 and 3, the heads of the two other mutts excluding the mutts of the plaintiffs, accepted the plaintiffs\' case to be correct. The contesting defendant, the State of Madras, denied that the Shri Krishna Shrine is just a temple attached to the ancient must started by Shri Madvacharya. It is stated that it is one of the most important temples in India, that it attracts a large body of worshippers from throughout the country, that the Hindu public worship there as of right and that considerable offerings are made to the temple. It is denied that the temple is intended only for the worship of the eight swamiyars. The learned Judges of the High Court delivered separate Judgments, but, in the main, agreed in the finding that the shrine in suit was a temple within the meaning of s. 2(1) of the Act, in view of the following considerations: (1) This institution had been referred to, by authoritative authors and in judicial pronouncements as Krishna temple. (2) The plaintiffs themselves stated in sub-paragraph (7) of paragraph 3 of the plaint that the Pariyayam Swami had the incontestable right to exclude anybody, including even the other Swamiyars of the eight mutts from the Shri Kishna Dev Temple during the period of his pariyayam. (3) The various documents by which certain properties had been endowed to this institution, refer to it as the temple. (4) The statements of the witnesses for the plaintiffs make it clear that Hindus in general, without any restriction, worship the deity, that different individuals had endowed permanent sevas, that the pilgrims include persons other than the followers of Shri Madvacharya and that during certain utsavs, thousands of Madvas and non-Madvas, Brahmins and non-Brahmins, attend the festival and the pilgrims put their money offerings in a box in front of the deity, (5) The inscriptions within the temple and the buildings near about show that this was not a private shrine appurtenant to the Mutt, but was a public institution under the general supervision of the Hindu sovereigns of that area. (6) The pamphlet issued for the help of the pilgrims, Exhibit B-10 states that all the pilgrims are invited to participate in the worship of the deity, which is done as many as nine times a day, and to make offerings. It gives details of the various kinds of sevas which could be offered by the devotees and the fees for each are to be determined by the Pariayayam Swami. It also states that the Pariyaya Mutt would arrange for homas, Udyapana, Thulabhara, Brahmopadesa, Annaprasana and Namakarana ceremonies for the benefit of the devotees on payment of the perscribed amounts and that the devotees could endow sums of money for burning Nanda Deepa and Deepa Sthamba throughout the year, (7) The predominant feature of the institution has been held to be the service or worship of Shri Krishna and not the propagation of religious knowledge of the system of philosophy propounded by Shri Madvacharya, The learned Judges were of opinion that the facts that the system of worship in this temple, is different from the system in other temples that persons visiting the Pariyayam festival are fed and religious discourses are given on the occasion and that the swamiyars of the eight mutts had been interred in the compound near the temple, did not adversely affect the conclusion from the other circumstances that the Shrine is a temple to which the Hindus in general could go for worship as a matter of right. On one point regarding Shri Madvacharya having a Mutt of his own, the learned Judges of the High Court appear to have had different views. Govinda Menon J., said: "It is seen that the present institution is far famed throughout the length and breadth of India and the large concourse of people resort to this place for worship and there is nothing to show that the Swamiars have treated this institution as their private property or anything of the kind. The admissions of P.W, 2 as well as the other witnesses point to the fact that the temple is one intended for 195 the benefit of the public and that public resort to it for religious worship and that though the image of Sri Krishna was installed by Madhwacharya in his own residence or lodging subsequent course of conduct during the centuries have made this abode of Sri Krishna a public temple." He also stated, at another place in his judgment, "As I have stated already, originally there was a mutt and subsequent to the installation of the image of Lord Krishna there by Sri Madhwacharya it has grown up by centuries of public worship into a public institution." Ramaswami J., has expressed himself differently on the question that Sri Madhvacharya had a mutt and installed the deity in it. He states, when considering the statement in Chapter 9 of the Madhwa Vijaya to the effect that Sri Madvacharya sportively brought the image to the Mutt, "The Math referred to in the 42nd stanza must obviously mean the Kshetra of Anantheswara which is referred to as Roopya in Sampradaya Paddhati of Hrishikesha Thirtha. The reason is, as well pointed out by the learned Subordinate Judge, that as Achyuta Preksha had taken his abode in that Kshetra and was engaged in teaching to his disciples therein, this Kshetra must have been referred to as the Math as by that time it is nowhere established that Madhwacharya had established a Math of his own and the teaching should have all been done in the temple of Anantheswara itself, Therefore, the Math referred to must be the temple of Anantheswara. This receives corroboration in two ways. In the end of Madhwa Vijayam, it is stated that while expounding Aittariyopanishat to his Sishyas in the temple of Anantheswara, the Acharya is stated to have disappeared from mundane existence. In fact even today the seat which is said to have been used by Sri Madhwacharya is shown to the pilgrims. Secondly, that the Anantheswara temple was the seat of teaching by Madhwacharya receives corroboration and therefore separate Krishna Math could not have existed is seen from the importance attaching to Anantheswara even today. Anantheswara is considered to be the holder of the Muli right and Sri Krishna is treated as his tenant." Learned counsel for the appellants submitted that the reference to the institution as a temple by the authoritative authors and in judicial pronouncements was really in a general sence, the institution being a temple in the usual sence of the terms as it is a shrine in which the idol of Sri Krishna is installed and is worshipped. His contention is that it is not a temple as defined in the Act, as the Hindus in general, or any section of the Hindus, had no right to worship there and that the fact that a number of pilgrims ordinarily visit the temple for worship is not inconsistent with the temple being a private one and not dedicated to the Hindus in general or a section of them. He pointed out that Ramaswami J., was wrong in stating: " P.W.1, the Matadhipathi of the Bhandrikere Mutt, had to admit in crossexamination the public participation as of right in the worship of the temple without the permission of the Paryaya Swami, thereby giving they go-by to the plaint allegations to make out that this is a private chaple." This is correct, but does not materially affect the decision as such a conclusion from the statement could be arrived at. We agree with the view of the learned Judges of the High Court that the shrine in suit is a temple as defined in s. 2 (1) of the Act. The evidence on record is fully consistent with the findings of the Courts below that this temple is a place dedicated to the Hindu public and is used by them as a place of public religious worship. It is not disputed that a large number of pilgrims from all over the country visit this place, take part in the worship there, make offerings to the deity and receive the prasad. The institution also receives monetary aid from the State. In Vibhudapriya v. Lakshmindra (1) is quoted at page 232, an extract from the South Kanara Manual published under the authority of Government. The extract reads: "The temple of Krishna, at Udipi, is said to have been founded by Madhavacharya him-self who set up in it the image of Krishna originally made by Arjuna and miraculously obtained from a vessel wrecked on the coast of Tuluva Besides the temple at Udipi he established eight Mathas\' or sacred houses, each presided over by a Sanyasi or Swami. These exist to this day and each Swami in turn presides over the temple of Krishna for a period of two years and spends the intervening fourteen years touring throughout Kanara and the adjacent parts of Mysore levying contributions from the faithful for the expenses of his next two years of office, which are very heavy as he has to defray not only the expenses of the public worship and of the temple and Matha establishments, but must also feed every Brahman who comes to the place." It is clear from this extract that the various Swamis tour about the country realising contributions from the devotees for the expenses which each of them has to incur during the period of his pariyayam, that the expenses which he has to incur during the period are heavy. The expenses are met out of the income during the two years of his pariyayam from the State aid and the offerings of pilgrims and income of his own mutt. The fact of raising contributions from the devotees resident in different parts of the country is clear proof of the fact that such devotees have a right to visit the temple and to worship there. If they have no such right, it is improbable that they would be visited by the Swamis for contributions. The fact that no instance of any pilgrim being refused permission to worship during the course of the centuries since the installation of the deity goes a long way in establishing and supporting the finding of the Court below that the institution has been held out as one for the benefit of the Hindus and that the Hindus in general have gone to it for worship in exercise of their right of worship and not on account of the permission, express or implied, of the Pariyayam Swami. It is true that the fact that a number of pilgrims visit the temple for worship regularly need not, in all cases, lead to the conclusion that the temple is a public one; but such a conclusion will not be arrived at only when there is good evidence about the temple being a private one. This is really what was held in [Babu Bhagwan Din v. Gir Har Saroop] (1) to which reference has been made by the learned counsel. The Privy Council, [in that case], after coming to the conclusion that the general effect of the evidence was that the family had treated the temple as family property, dividing the various forms of profit, whether offerings or rents, closing it so as to exclude the public from worship when marriage or other ceremonies required the attendance of the members of the family at its original home and erecting samadhs to the honour of its, dead, said at page 9: "In these circumstances, it is not enough in their Lordships\' opinion, to deprive the family of their private property to show that Hindus willing to worship have never been turned away, or even that the deity has acquired considerable popularity among Hindus of the locality or among persons resorting to the annual mela. Worshippers are naturally welcome at a temple because of the offerings they bring and the repute they give to the idol; they do not have to be turned away on pain of forfeiture of the temple property as having become property belonging to a public trust. Facts and circumstances, in order to be accepted as sufficient proof of dedication of a temple as a public temple, must be considered in their historical setting in such a case as the present; and dedication to the public is not to be readily inferred when it is known that the temple property was acquired by grant to an individual or family." It follows, therefore, that in the absence of goon evidence that a temple is a private one, the mere fact that it is visited by a large number of persons among the Hindu public without any restrain for a number of years, will be good evidence of the fact that the temple had been dedicated to the Hindu public and was for its benefit. Reference may usefully be made to the case reported as Sri Venkataramana Devaru, v. The State of Mysore. (1) In this case, a temple was founded for the benefit of Gowda Saraswath Brahmins, who managed it throughout. They were the followers of the Kashi Mutt. The head of that Mutt performed various religious ceremonies in the temple. It was alleged that persons who were not Gowda Saraswath Brahmins could not enter without the permission of the trustees. However, there was no instance in which such permission was refused. There was evidence that all communities had been freely admitted into the temple. It was contended that the free admission of all communities and there being no instance of any refusal of permission, led to the conclusion that the Hindu public generally had a right to worship in the temple. In considering this contention, it was said at page 907: "The law on the subject is well settled. When there is a question as to the nature and extent of a dedication of a temple, that has to be determined on the terms of the deed of endowment if that is available, and where it is not, on other materials legally admissible; and proof of long and uninterrupted user would be cogent evidence of the terms thereof. Where, therefore, the original deed of endowment is not available and it is found that all persons are freely worshipping in the temple without let or hindrance, it would be a proper inference to make that they do so as a matter of right, and that the original foundation was for their benefit as well. But where it is proved by production of the deed of endowment or otherwise that the original dedication was for the benefit of a particular community, the fact that members of other communities were allowed freely to worship cannot lead to the inference that the dedication was for their benefit as well. For, as observed in Babu Bhagwan Din v. Gir Har Saroop (67 I.A. 1), it would not in general be consonant with Hindu sentiments or practice that worshippers should be turned away\'." There is no documentary evidence in this case for supporting the contention of the appellants that the temple was originally founded for the private use of Shri Madvacharya and his disciples. In the absence of such evidence, the long user of the temple by the Hindus in general, together with there being no instance of anybody having been refused permission, must lead to the conclusion and support the finding that the temple had been dedicated to the Hindus in general, and was for their benefit. Further, there is no evidence on record, oral or documentary, of course oral was not possible, of the fact that Shri Madvacharya had a Mutt of his own prior to his obtaining the idol of Sri Krishna which he installed in this temple. He is said to have set up eight different Mutts, each for one of his eight disciples. All these eight Mutts have particular names. No Mutt is named after Sri Madvacharya. Ramaswami J., has given good reasons for the view that Sri Madavcharya had no Mutt of his own. A primary Mutt associated with the founder himself must have and independent permanent head. There is no such Matathipathi or head of this so-called Mutt. One of the eight swamis, the heads of the Asth Mutts, acts as head or manager of this institution for a period of two years. The absence of a head and this system of a head or manager being appointed by rotation, very clearly point to the conclusion that the institution in suit is neither a Mutt nor a temple appurtenent to a Mutt. In 1937, the Board of Commissioners for the Hindu Religious Endowments, Madras, passed an order under s. 84 of the Madras Hindu Religious Endowments Act, 1926 (Act II of 1927), that this institution was not a temple as defined in that Act, but was a place of worship appurtenant to Sri Krishna Devaru Math, Shivalli, Udipi Taluk, South Kanara, District. It has been urged for the appellant that this order bars any further enquiry and a different conclusion under the Act with respect to the nature of this institution. The Courts below have held against this contention and, we think, rightly. The finding of the Board was, in their own words: "Our decision that the institution is appurtenant to a math and forms part of it can in no wise affect the rights of the deity to the properties owned by it and the rights of the Hindu public to worship direct, subject to the regulations prescribed by the Paryayam Swamiar for the time being. We hold that it is not a temple as defined in the Act, but it is a place of worship appurtenant to the Math." The finding that the Hindu public had a right to worship in this temple is sufficient to make the institution a temple within the definition of that term in the Act (Madras Act V of 1947), even if the temple be appurtenant to a Mutt. The Board\'s order, therefore, cannot affect the consideration of the question of the institution being a temple within the meaning of the definition in the Act. We are therefore of opinion that the Courts below have come to the correct conclusion that the institution in suit is a temple within the meaning of the word "temple" in the Act and that the appeal should fail. We accordingly dismiss the appeal with costs. Appeal dismissed.', 'word_count': 4269, 'text_length_chars': 24257}  --- Entry 2 --- {'title': 'Shri Mahadeo Paikaji Kolhe Yavatmal vs The State Of Bombay', 'date': '1961-04-04', 'court': 'Supreme Court of India', 'citations': '1961 AIR 1517, 1962 SCR (1) 733, AIR 1961 SUPREME COURT 1517', 'bench': 'P.B. Gajendragadkar , A.K. Sarkar , K.N. Wanchoo , K.C. Das Gupta , N. Rajagopala Ayyangar', 'headnote': "The petitioners challenged the constitutional validity of the Bombay Tenancy and Agricultural Lands (Vidarbha Region and Kutch Area) Act, 1958 , which extended the provisions of the Bombay Tenancy and Agricultural Lands (Amendment) Act, 1956, to Vidarbha and Kutch. That Act was declared valid by this Court in Sri Ram Narain Medhi v. The State of Bombay, [1950] Supp. 1 S.C.R. 489, and one of the reason-, for doing so was that the lands covered by that Act fell within the definition of the word 'estate' contained in the Bombay Land Revenue Code, 1879. The lands in question in the present petitions were situated in Amraoti and Yeotmal and the existing law relating to land tenures in force in that area was the Madha Pradesh Land Revenue Code, 1954. This Code did not employ the word ,estate' and it was contended by the petitioners that the impugned Act was not within the protection of Art. 31A of the Constitution. Held, that the contention must fail. Although the Madhya Pradesh Land Revenue Code, 1954, did not employ the word 'estate', the relevant definition contained in ss. 2(17) and 2(18) of impugned Act and ss. 2(7) , 2(20) of the Code read with ss. 145 and I46 thereof leaves no manner of doubt that the lands in the possession of the petitioners were tenures and in substance ,in estate. Since the petitioners held the lards tinder the State and paid land revenue for them, the lands fell within the class of local equivalents of the word 'estate.' as contemplated by Art. 31A(2)(a) of the Constitution.", 'text': 'PETITIONER: SHRI MAHADEO PAIKAJI KOLHE YAVATMAL Vs.\n\nRESPONDENT: THE STATE OF BOMBAY\n\nDATE OF JUDGMENT: 04/04/1961\n\nBENCH: GAJENDRAGADKAR, P.B.\n\nBENCH: GAJENDRAGADKAR, P.B. SARKAR, A.K. WANCHOO, K.N. GUPTA, K.C. DAS AYYANGAR, N. RAJAGOPALA\n\nCITATION: 1961 AIR 1517 1962 SCR (1) 733\n\nCITATOR INFO: R 1962 SC 694 (28,67) R 1967 SC1110 (11) R 1970 SC 439 (14)\n\nACT: Agricultural Land-Amendment of law relating to tenancies- Constitutional validity of enactment Bombay Tenancy and Agricultural Lands (Vidarbha Region and Kutch Area) Act, 1958 (Bom. 99 of 1958).\n\nHEADNOTE: The petitioners challenged the constitutional validity of the Bombay Tenancy and Agricultural Lands (Vidarbha Region and Kutch Area) Act, 1958, which extended the provisions of the Bombay Tenancy and Agricultural Lands (Amendment) Act, 1956, to Vidarbha and Kutch. That Act was declared valid by this Court in Sri Ram Narain Medhi v. The State of Bombay, [1950] Supp. 1 S.C.R. 489, and one of the reason-, for doing so was that the lands covered by that Act fell within the definition of the word \'estate\' contained in the Bombay Land Revenue Code, 1879. The lands in question in the present petitions were situated in Amraoti and Yeotmal and the existing law relating to land tenures in force in that area was the Madha Pradesh Land Revenue Code, 1954. This Code did not employ the word,estate\' and it was contended by the petitioners that the impugned Act was not within the protection of Art. 31A of the Constitution. Held, that the contention must fail. Although the Madhya Pradesh Land Revenue Code, 1954, did not employ the word \'estate\', the relevant definition contained in ss. 2(17) and 2(18) of impugned Act and ss. 2(7), 2(20) of the Code read with ss. 145 and I46 thereof leaves no manner of doubt that the lands in the possession of the petitioners were tenures and in substance,in estate. Since the petitioners held the lards tinder the State and paid land revenue for them, the lands fell within the class of local equivalents of the word \'estate.\' as contemplated by Art. 31A(2)(a) of the Constitution.\n\nJUDGMENT:\n\nORIGINAL JURISDICTION: Writ Petitions Nos. 93 and 125 of\n\n1959. Petitions under Art. 32 of the Constitution of India for enforcement of Fundamental Rights. V. M. Limaye, E. Udayarathnam and S. S. Shukla, for the petitioners. H.N. Sanyal, Additional Solicitor-General of India, R. Ganapathy Iyer and D. Gupta, for the respondent. W. S. Barlingay and A. G. Ratnaparkhi, for the Interveners.\n\n1961. April 4. The Judgment of the Court was delivered by GAJENDRAGADKAR, J.-These two writ petitions have been filed under Art. 32 of the Constitution and they seek to challenge the validity of the [Bombay Tenancy and Agricultural Lands Act] 99 of 1958 (hereafter called the Act). The impugned Act in substance is intended to extend to Vidarbha region and Cutch area which had then become a part -of the Greater Bi- Lingual State of Bombay the provisions of the Bombay Tenancy and Agricultural Lands (Amendment).Act, 1956 (Act XIII of 1956). The preamble to the impugned Act shows that it was intended to amend the law relating to tenancies of agricultural lands and sites used for allied purposes in the two areas of the State of Bombay and to make certain other provisions in regard to those lands. In extending the provisions of the earlier Bombay Act XIII of 1956 to the two areas the legislature has conformed to the pattern set up by the said earlier Act. The policy underlying the Act and the object intended to be achieved by it are the same and the method adopted in achieving that object is also the same. The validity of the earlier Bombay Act (XIII of 1956) was challenged before this Court in [Sri Ram Ram Narain Medhi v. The State of Bombay] (1) but the challenge failed and the Act was held to be constitutional. One of the points which arose for decision in that case was whether the impugned Act was protected by Art. 31A(2)(a) of the Constitution, and the answer to that question depended upon the determination of another issue which was whether the lands to which the said Act applied were an "estate" as required by Art. 31A(2)(a). In dealing with that question this Court held that the word "estate" as defined by s. 2(5) of the Bombay Land Revenue Code, 1879, clearly applied to the lands\n\n(1) [1959] Supp. 1 S.C.R. 489. covered by the Act and so Art. 31A(2)(a) was applicable. Having regard to this decision the only point which Mr. Limaye attempted to raise before us in support of the two writ petitions is that the lands belonging to the two petitioners are not an "estate" within the meaning of [Art. 31A(2)(a)], and so the impugned Act is outside the protection of Art. 31A. If this contention is not upheld then it is obvious that(the writ petitions must fail; if the said contention is upheld then of course the other contentions raised by the two writ petitions against the validity of certain specific provisions of the Act may fall to be considered. The two petitioners are Namdeorao Baliramji and Mahadeo Paikaji Kolhe respectively. The first one resides at Amraoti and the second at Yeotmal. The first owns about 80 acres dry lands situated in Amraoti out of which 43 acres are under his personal cultivation and the rest in the possession of the tenants. The second petitioner owns about 1168 acres dry lands situated in Yeotmal out of which 400 acres are under his personal cultivation and the rest with the tenants. The lands in both the cases are charged to the payment of land revenue. The case for both the petitioners is that the lands thus held by them are not an "estate" within the meaning of Art. 31A(2)(a). Article 31A(2)(a) provides, inter alia, that the expression "estate" shall in relation to any local area have the same meaning as that expression or its local equivalent has in the existing law relating to land tenures in force in that area. The existing local law, it is common ground, is the Madhya Pradesh Land Revenue Code, 1954 (II of 1955), and so it is necessary to find whether the lands belonging to the petitioners can be said to be an \'estate\' within the meaning of the said Code. Before we do so, however, it may be pertinent to refer to the relevant definitions in the impugned Act. Section 2(17) of the Act defines land as meaning, inter alia, land which is used or capable of being used for agricultural purposes and includes the sites of farm buildings appurtenant to such land. Section 2(18) defines a land-holder as meaning a tenure-holder whom the State Government has declared on account of tile extent and value of the, land or his interests therein to be a land-holder for the purposes of this Act. Now,s.27 of the Madhya Pradesh Land Revenue Code in question defines a holding as meaning, inter alia,a parcel of land separately assessed to land revenue,and s. 2(20) defines a tenure-holder as meaning a person holding from the State Government as a Bhumiswami or a Bhumidari. Chapter XII of the Code deals with tenure-holders. Section 145 provides that there, shall be two classes of tenure,holders of lands held from the State, namely, Bhumiswami and Bhumidhari. Section 146 deals with Bhumiswami. It provides that "every person who at the coming into force of this Code belongs to any of the classes specified in clauses (a) to (f) of the said section shall be called a Bhumiswami and shall have all the rights and is subject to all the liabilities conferred or imposed upon a Bhumiswami by or under this Code". Amongst these classes is the class covered by el. (e) which relates to persons in respect of lands held by them as occupants in Berar. Thus reading the relevant definitions along with the provisions of s. 146 of the Code it would follow that the land in the possession of the Bhumiswami who is a tenure-holder is in substance all estate. It is true that the word "estate" as such has not been employed in the Code, but it must be borne in mind that Art. 31A(2)(a) refers not only to estate but also to its local equivalent. It was realised that in many areas the existing law relating to land tenures may not expressly define all estate as such though the said areas had their local equivalents described and defined. That is why the relevant provision of the Constitution has deliberately used both the word "estate" as well as its local equivalent".The petitioners hold lands under the State and they pay land revenue for the, lands thus held by them.Therefore, there is no difficulty in holding that under the existing law relating to land tenures the lands held by them fill within the class of the local equivalents of the word "estate" as contemplated by Art. 31A(2)(a). If that is so the contention raised by Mr. Limaye that the impugned Act is not protected by Art. 31A cannot succeed. As we have already indicated it is not disputed that if Art. 31A applies there can be no further challenge to the validity of the impugned statute. The writ petitions accordingly fail and are dismissed with costs one SEt of hearing costs. Petitions dismissed.', 'word_count': 1555, 'text_length_chars': 9036}  
now what should I do to streamline the process and train a byte level bpe on it fast, with roughly the same number of entries from each file

In [ ]:
import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


In [ ]:
# Upload Trained Tokenizer onto Hugging Face Hub
from huggingface_hub import HfApi, create_repo

def upload_tokenizer_to_hf(tokenizer_dir: str, repo_id: str):
    if not HF_TOKEN:
        print("No HF_TOKEN found, skipping upload.")
        return
    print(f"\nPreparing to upload tokenizer files to Hugging Face Hub repo: '{repo_id}'...")
    api = HfApi(token=HF_TOKEN)

    try:
        # create_repo returns the fully qualified "username/name" id; a bare name would fail in upload_file
        repo_id = create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True, repo_type="model").repo_id
        print(f"  Hugging Face repository '{repo_id}' is ready.")

        files_to_upload = [f for f in os.listdir(tokenizer_dir) if os.path.isfile(os.path.join(tokenizer_dir, f))]
        for file_name in tqdm(files_to_upload, desc="Uploading files to HF"):
            api.upload_file(path_or_fileobj=os.path.join(tokenizer_dir, file_name),
                            path_in_repo=file_name, repo_id=repo_id, repo_type="model")

        print(f"\nSUCCESS: Tokenizer uploaded to https://huggingface.co/{repo_id}")
    except Exception as err:
        print(f"Notice during HF Upload: {err}")
        print("To upload to your own HF account, set HF_TOKENIZER_UPLOAD_REPO = 'your_username/your_repo_name'.")

# upload_tokenizer_to_hf(TOKENIZER_DIR, HF_TOKENIZER_UPLOAD_REPO)

In [ ]:
# Sample Balanced Representative Texts Across Files with tqdm
from tqdm.auto import tqdm
import os
from typing import List
import ijson
from tqdm.auto import tqdm

CHARS_PER_FILE = 20_000_000

def extract_training_sample(file_paths: List[str], chars_per_file: int = CHARS_PER_FILE) -> List[str]:
    sampled_texts = []
    print(f"\nExtracting ~{chars_per_file:,} chars from each file for tokenizer merge learning...")

    for path in file_paths:
        fname = os.path.basename(path)
        seen = 0
        try:
            with open(path, "rb") as f, tqdm(total=chars_per_file, unit="char", unit_scale=True, desc=fname) as pbar:
                for rec in ijson.items(f, "item"):
                    if not isinstance(rec, dict):
                        continue
                    text = (rec.get("text") or "").strip()
                    if text:
                        sampled_texts.append(text)
                        seen += len(text)
                        pbar.update(len(text))
                    if seen >= chars_per_file:
                        break
        except Exception as err:
            print(f"Error reading {fname}: {err}")
        if seen < chars_per_file:
            print(f"  WARNING: {fname} only gave {seen:,} chars")

    print(f"Extracted {len(sampled_texts):,} documents for tokenizer training.")
    return sampled_texts

# training_corpus = extract_training_sample(train_paths)

In [ ]:
# Train Pure Byte-Level BPE Tokenizer from Scratch

from huggingface_hub import HfApi, hf_hub_download
TOK_FILES = ["vocab.json", "merges.json", "tokenizer_config.json"]
HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase1"

def get_tokenizer():
    if all(os.path.exists(os.path.join(TOKENIZER_DIR, f)) for f in TOK_FILES):
        print("Loaded tokenizer from disk");  return PureByteLevelBPETokenizer.load(TOKENIZER_DIR)
    try:
        for f in TOK_FILES:
            hf_hub_download(HF_TOK_REPO, f, repo_type="model", local_dir=TOKENIZER_DIR, token=HF_TOKEN)
        print("Loaded tokenizer from HF"); return PureByteLevelBPETokenizer.load(TOKENIZER_DIR)
    except Exception as e:
        print("No saved tokenizer found:", e); return None

tokenizer = get_tokenizer()
if tokenizer is None: # only runs the very first time
    training_corpus = extract_training_sample(train_paths)
    tokenizer = PureByteLevelBPETokenizer(VOCAB_SIZE)
    tokenizer.train_from_texts(training_corpus, target_vocab_size=VOCAB_SIZE)
    tokenizer.save(TOKENIZER_DIR)
    upload_tokenizer_to_hf(TOKENIZER_DIR, HF_TOK_REPO)

In [ ]:
# Rigorous Tokenizer Sanity Checks
print("\n================ SANITY CHECK REPORT ================")

# 1. Roundtrip Fidelity Test
test_sentence = "The Madras Temple Entry Authorization Act, 1947: PETITIONER vs RESPONDENT in Civil Appeal No. 84 of 1959."
encoded_ids = tokenizer.encode(test_sentence)
decoded_sentence = tokenizer.decode(encoded_ids)

print(f"[Check 1] Original String : '{test_sentence}'")
print(f"[Check 1] Token Count     : {len(encoded_ids)} tokens")
print(f"[Check 1] Decoded String  : '{decoded_sentence}'")
assert test_sentence == decoded_sentence, "ERROR: Roundtrip decode did not match original string!"
print("  -> PASS: 100% Exact Roundtrip Match!")

# 2. Special Tokens Validation
print(f"[Check 2] Special Tokens  : BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}, UNK={tokenizer.unk_token_id}")
assert len(set([tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, tokenizer.unk_token_id])) == 4, "ERROR: Duplicate Special Token IDs!"
print("  -> PASS: All Special Tokens Unique & Bound!")

# 3. Save & Load Reload Check
reloaded_tokenizer = PureByteLevelBPETokenizer.load(TOKENIZER_DIR)
reloaded_encoded = reloaded_tokenizer.encode(test_sentence)
assert encoded_ids == reloaded_encoded, "ERROR: Reloaded tokenizer encoding mismatch!"
print("  -> PASS: Save/Load State Integrity Verified!")

# 3b. Reload check on awkward characters (spaces, newlines, non-ASCII all appear inside tokens)
hard = "Held:  that\n\nSection 12 \u00a7 3\r\n\u0939\u093f\u0902\u0926\u0942 appellant."
assert reloaded_tokenizer.decode(reloaded_tokenizer.encode(hard)) == hard, "ERROR: hard-string roundtrip failed"
assert len(reloaded_tokenizer.merges) == len(tokenizer.merges), "ERROR: merges lost on reload"
print("  -> PASS: Hard-string roundtrip + all merges reloaded!")

# 4. Compression Ratio Analysis
sample_bytes = len(test_sentence.encode('utf-8'))
compression = sample_bytes / len(encoded_ids)
print(f"[Check 4] Compression     : {sample_bytes} bytes -> {len(encoded_ids)} tokens ({compression:.2f} bytes/token)")
print("====================================================\n")

In [ ]:
def iter_sampled(path, n_docs, seed=42, val_frac=0.01, max_scan=None):
    total = min(KNOWN_COUNTS[os.path.basename(path)], max_scan or 10**12)
    rng = random.Random(seed)
    chosen = set(rng.sample(range(total), min(n_docs, total)))
    last = max(chosen)
    with open(path, "rb") as f:
        for i, rec in enumerate(ijson.items(f, "item")):
            if i > last: break
            if i in chosen:
                t = (rec.get("text") or "").strip()
                if t: yield (rng.random() < val_frac, t)

_TOK = None
def _init(d):
    global _TOK; _TOK = PureByteLevelBPETokenizer.load(d)
def _enc(item):
    is_val, t = item
    ids = _TOK.encode(t); ids.append(_TOK.eos_token_id)
    return is_val, np.array(ids, dtype=np.uint16)

DOMAIN = {"train_acts.json": "acts", "train_supreme_court.json": "supreme_court",
               "train_high_courts.json": "high_courts", "train_tribunals.json": "tribunals"}

def build_bins(path, n_docs, out_dir=TOKEN_DATA_DIR, **kw):
    dom = DOMAIN[os.path.basename(path)]
    os.makedirs(out_dir, exist_ok=True)
    got = 0
    with open(f"{out_dir}/train_{dom}.bin", "wb") as ftr, open(f"{out_dir}/val_{dom}.bin", "wb") as fva, \
         Pool(4, initializer=_init, initargs=(TOKENIZER_DIR,)) as pool:
        for is_val, arr in tqdm(pool.imap(_enc, iter_sampled(path, n_docs, **kw), chunksize=8)):
            (fva if is_val else ftr).write(arr.tobytes())
            got += 0 if is_val else len(arr)
    return got

N_DOCS = {
    "train_acts.json": 50_000,
    "train_supreme_court.json": 25_000,
    "train_high_courts.json": 25_000,
    "train_tribunals.json": 25_000}

if not os.path.exists(f"{TOKEN_DATA_DIR}/train_tribunals.bin"):
    for p in train_paths:
        fn = os.path.basename(p)
        print(fn, build_bins(p, N_DOCS[fn]), "train tokens")


In [ ]:
print(type(N_DOCS))
print(N_DOCS)

In [ ]:
# code before this is correct. after this, not so much

In [ ]:
# to do
# should use both gpus and all 4 cpus, besides that, should also show both val and train loss and check how u count model params
# earlier, you should not train vocab every time, so import when needed
# attention is all you need results: regularization, label smoothing, residual dropout

I do not wanna use a GPT-2 style model because it has learned positional encodings, performs poorly
I'd rather try rotary or alibi
besides that, I also wanna try

In [ ]:
import random, ijson

tok = get_tokenizer()
show = lambda ids: [tok.decoder[i].decode("utf-8", "replace") for i in ids]


print("vocab size :", len(tok.encoder), "(target", VOCAB_SIZE, ")")
print("merges     :", len(tok.merges), "| expected ~", VOCAB_SIZE - 256 - len(tok.special_tokens))
assert sorted(tok.decoder) == list(range(len(tok.decoder))), "ids are not contiguous 0..N-1"
assert len(set(tok.encoder.values())) == len(tok.encoder), "duplicate ids"
print("specials   :", dict(pad=tok.pad_token_id, unk=tok.unk_token_id, bos=tok.bos_token_id, eos=tok.eos_token_id))


print("\n30 longest tokens :", [repr(t) for t in show(sorted(tok.decoder, key=lambda i: -len(tok.decoder[i]))[:30])])
print("20 earliest merges:", [repr(tok.merges[p].decode("utf-8", "replace")) for p in list(tok.merges)[:20]])
print("20 latest merges  :", [repr(tok.merges[p].decode("utf-8", "replace")) for p in list(tok.merges)[-20:]])
print("20 random tokens  :", [repr(t) for t in show(random.sample(range(261, len(tok.decoder)), 20))])


s = "The appellant filed a writ petition under Article 226 of the Constitution before the High Court."
print("\n", show(tok.encode(s)))


for p in train_paths:
    with open(p, "rb") as f:
        docs = [r["text"] for _, r in zip(range(200), ijson.items(f, "item")) if r.get("text")]
    ids = [tok.encode(d) for d in docs]
    n_tok, n_byte = sum(map(len, ids)), sum(len(d.encode("utf-8")) for d in docs)
    n_unk = sum(i.count(tok.unk_token_id) for i in ids)
    n_single = sum(1 for x in ids for i in x if len(tok.decoder[i]) == 1)
    print(f"{os.path.basename(p):28s} bytes/token={n_byte/n_tok:.2f}  unk={n_unk}  single-byte tokens={100*n_single/n_tok:.1f}%")



In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
%%writefile train_llm.py
"""
Pretraining script for the handwritten Llama-style model (llama_scratch.py).

Launch (2 GPUs):   torchrun --standalone --nproc_per_node=2 train_llm.py [--bench] [--resume]

- fp16 autocast + GradScaler (T4 has no bf16), grad accumulation, DDP across both GPUs
- every sequence is drawn from a domain chosen with probability 40/20/20/20 (acts / supreme / high courts / tribunals)
- the LR schedule follows max(step progress, wall-clock progress), so cosine decay always finishes inside the time budget
- logs train loss every few steps and per-domain + weighted validation loss every --eval_every steps
"""
import argparse
import json
import math
import os
import time
from tqdm.auto import tqdm
from contextlib import nullcontext
from dataclasses import asdict

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from llama_scratch import Config, PRESETS, LlamaLM, count_params

DOMAIN_MIX = {"acts": 0.4, "supreme_court": 0.2, "high_courts": 0.2, "tribunals": 0.2}


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", default="/kaggle/working/cs781_phase1/token_data")
    p.add_argument("--out_dir", default="/kaggle/working/cs781_phase1/model_checkpoint")
    p.add_argument("--preset", default="base")
    p.add_argument("--pos_enc", default="rope", choices=["rope", "alibi"])
    p.add_argument("--attn_impl", default="manual", choices=["manual", "sdpa"])
    p.add_argument("--seq_len", type=int, default=1280)
    p.add_argument("--micro_bs", type=int, default=0, help="per-GPU sequences per forward; 0 = auto (2 manual / 4 sdpa)")
    p.add_argument("--global_bs", type=int, default=128, help="sequences per optimizer step across all GPUs")
    p.add_argument("--total_tokens", type=float, default=200e6)
    p.add_argument("--time_hours", type=float, default=4.5, help="hard wall-clock budget for the training loop")
    p.add_argument("--lr", type=float, default=6e-4)
    p.add_argument("--min_lr_frac", type=float, default=0.1)
    p.add_argument("--warmup_steps", type=int, default=100)
    p.add_argument("--wd", type=float, default=0.1)
    p.add_argument("--clip", type=float, default=1.0)
    p.add_argument("--log_every", type=int, default=0)
    p.add_argument("--eval_every", type=int, default=0)
    p.add_argument("--eval_batches", type=int, default=16)
    p.add_argument("--ckpt_every", type=int, default=0)
    p.add_argument("--seed", type=int, default=1337)
    p.add_argument("--bench", action="store_true", help="run ~15 steps, print throughput, exit")
    p.add_argument("--resume", action="store_true")
    return p.parse_args()


class DomainSampler:
    """Random windows of seq_len+1 tokens from per-domain uint16 memmaps, domain picked by weight."""

    def __init__(self, data_dir, split, names, weights, seq_len, seed, device):
        self.arrs = [np.memmap(os.path.join(data_dir, f"{split}_{n}.bin"), dtype=np.uint16, mode="r") for n in names]
        for n, a in zip(names, self.arrs):
            assert len(a) > seq_len + 2, f"{split}_{n}.bin has only {len(a)} tokens"
        w = np.array(weights, dtype=np.float64)
        self.w = w / w.sum()
        self.T = seq_len
        self.rng = np.random.default_rng(seed)
        self.device = device

    def batch(self, bs, only=None):
        T = self.T
        doms = [only] * bs if only is not None else self.rng.choice(len(self.arrs), size=bs, p=self.w)
        x = np.empty((bs, T), dtype=np.int64)
        y = np.empty((bs, T), dtype=np.int64)
        for i, d in enumerate(doms):
            a = self.arrs[d]
            s = int(self.rng.integers(0, len(a) - T - 1))
            chunk = a[s:s + T + 1].astype(np.int64)
            x[i], y[i] = chunk[:-1], chunk[1:]
        return (torch.from_numpy(x).to(self.device, non_blocking=True),
                torch.from_numpy(y).to(self.device, non_blocking=True))


def main():
    args = get_args()
    use_ddp = "RANK" in os.environ
    if use_ddp:
        dist.init_process_group("nccl")
        rank, world, local = int(os.environ["RANK"]), int(os.environ["WORLD_SIZE"]), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
    master = rank == 0
    torch.manual_seed(args.seed)          # same init on every rank

    names = list(DOMAIN_MIX.keys())
    weights = [DOMAIN_MIX[n] for n in names]

    micro_bs = args.micro_bs or (2 if args.attn_impl == "manual" else 4)
    assert args.global_bs % (world * micro_bs) == 0, "global_bs must be divisible by world*micro_bs"
    accum = args.global_bs // (world * micro_bs)
    tokens_per_step = args.global_bs * args.seq_len
    max_steps = int(args.total_tokens // tokens_per_step)
    args.log_every  = args.log_every  or max(1, max_steps // 100)
    args.eval_every = args.eval_every or max(1, max_steps // 15)
    args.ckpt_every = args.ckpt_every or max(1, max_steps // 10)
    time_budget = args.time_hours * 3600.0

    cfg = Config(**PRESETS[args.preset], max_seq_len=args.seq_len, attn_impl=args.attn_impl, pos_enc=args.pos_enc)
    model = LlamaLM(cfg).to(device)

    if master:
        os.makedirs(args.out_dir, exist_ok=True)
        print(f"GPUs: {world} | preset={args.preset} pos_enc={args.pos_enc} attn={args.attn_impl}")
        count_params(model)
        print(f"seq_len {args.seq_len} | micro_bs {micro_bs} x accum {accum} x {world} GPUs = {args.global_bs} seqs "
              f"= {tokens_per_step:,} tokens/step | max_steps {max_steps:,} | time budget {args.time_hours} h")
        for n in names:
            tr = np.memmap(os.path.join(args.data_dir, f"train_{n}.bin"), dtype=np.uint16, mode="r")
            top = int(tr[: min(len(tr), 5_000_000)].max())
            assert top < cfg.vocab_size, f"{n}: token id {top} >= vocab_size {cfg.vocab_size}"
            print(f"  {n:14s} unique train tokens {len(tr)/1e6:7.1f}M  -> will see ~{args.total_tokens*DOMAIN_MIX[n]/1e6:.0f}M "
                  f"({args.total_tokens*DOMAIN_MIX[n]/len(tr):.2f} epochs)")

    decay = [p for _, p in model.named_parameters() if p.dim() >= 2]
    no_decay = [p for _, p in model.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.wd},
                             {"params": no_decay, "weight_decay": 0.0}],
                            lr=args.lr, betas=(0.9, 0.95), eps=1e-8, fused=True)
    scaler = torch.amp.GradScaler("cuda")

    ckpt_path = os.path.join(args.out_dir, "ckpt.pt")
    start_step, elapsed_prev = 0, 0.0
    history = {"train": [], "val": []}
    if args.resume and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu")
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        start_step, elapsed_prev, history = ck["step"], ck["elapsed"], ck["history"]
        if master:
            print(f"resumed from step {start_step}, {elapsed_prev/3600:.2f} h already used")
        del ck

    best_val = min((v["weighted"] for v in history["val"]), default=float("inf"))
    def maybe_save_best(step, r):
        nonlocal best_val
        if r["weighted"] < best_val:
            best_val = r["weighted"]
            tmp = os.path.join(args.out_dir, "best_model.pt.tmp")
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "step": step, "val": r}, tmp)
            os.replace(tmp, os.path.join(args.out_dir, "best_model.pt"))
            tqdm.write(f"   new best weighted val {best_val:.4f} -> best_model.pt")

    ddp_model = DDP(model, device_ids=[local]) if use_ddp else model
    train_data = DomainSampler(args.data_dir, "train", names, weights, args.seq_len,
                               seed=1000 + rank + 7919 * start_step, device=device)
    val_data = DomainSampler(args.data_dir, "val", names, weights, args.seq_len, seed=0, device=device) if master else None

    @torch.no_grad()
    def evaluate():
        model.eval()
        res = {}
        for d, n in enumerate(names):
            val_data.rng = np.random.default_rng(1234 + d)            # same windows every evaluation
            tot = 0.0
            for _ in range(args.eval_batches):
                x, y = val_data.batch(micro_bs, only=d)
                with torch.autocast("cuda", dtype=torch.float16):
                    tot += model(x, targets=y).item()
            res[n] = tot / args.eval_batches
        res["weighted"] = sum(DOMAIN_MIX[n] * res[n] for n in names)
        model.train()
        return res

    def save_ckpt(step, elapsed):
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "elapsed": elapsed, "cfg": asdict(cfg), "history": history}, tmp)
        os.replace(tmp, ckpt_path)
        with open(os.path.join(args.out_dir, "history.json"), "w") as f:
            json.dump(history, f)

    def lr_at(step, progress):
        if step < args.warmup_steps:
            return args.lr * (step + 1) / args.warmup_steps
        coeff = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
        return args.lr * (args.min_lr_frac + (1.0 - args.min_lr_frac) * coeff)

    model.train()
    t0 = time.time() - elapsed_prev
    win_t, win_steps = time.time(), 0
    bench_t = None
    step = start_step
    pbar = tqdm(total=max_steps, initial=start_step, desc="train", unit="step",
            dynamic_ncols=True, mininterval=60, disable=not master)
    while True:
        elapsed = time.time() - t0
        prog = torch.tensor([max(step / max_steps, elapsed / time_budget)], device=device)
        if use_ddp:
            dist.broadcast(prog, 0)                      # all ranks must agree on LR and on when to stop
        progress = prog.item()
        if progress >= 1.0:
            break
        lr = lr_at(step, progress)
        for g in opt.param_groups:
            g["lr"] = lr

        opt.zero_grad(set_to_none=True)
        loss_sum = torch.zeros((), device=device)
        for m in range(accum):
            x, y = train_data.batch(micro_bs)
            sync_ctx = ddp_model.no_sync() if (use_ddp and m < accum - 1) else nullcontext()
            with sync_ctx:
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = ddp_model(x, targets=y)
                scaler.scale(loss / accum).backward()
            loss_sum += loss.detach() / accum
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
        scaler.step(opt)
        scaler.update()
        step += 1
        win_steps += 1

        if use_ddp:
            dist.all_reduce(loss_sum)
            loss_sum /= world

        if master:
            pbar.update(1)
            pbar.set_postfix(loss=f"{loss_sum.item():.3f}", lr=f"{lr:.1e}", refresh=False)

        if args.bench:
            if step == start_step + 4:
                torch.cuda.synchronize()
                bench_t = time.time()
            if step == start_step + 14:
                torch.cuda.synchronize()
                dt = (time.time() - bench_t) / 10
                tps = tokens_per_step / dt
                if master:
                    print(f"\nBENCH: {dt:.2f} s/step | {tps:,.0f} tokens/s total | "
                          f"peak mem {torch.cuda.max_memory_allocated()/2**30:.1f} GiB/GPU")
                    print(f"       {args.total_tokens/1e6:.0f}M tokens would take {args.total_tokens/tps/3600:.2f} h "
                          f"| in {args.time_hours} h you get ~{tps*args.time_hours*3600/1e6:.0f}M tokens")
                break

        if master and step % args.log_every == 0:
            torch.cuda.synchronize()
            now = time.time()
            tps = win_steps * tokens_per_step / (now - win_t)
            win_t, win_steps = now, 0
            history["train"].append([step, loss_sum.item()])
            tqdm.write(f"step {step:5d}/{max_steps} | loss {loss_sum.item():.4f} | lr {lr:.2e} | gnorm {gnorm.item():.2f} | "
                f"{tps:,.0f} tok/s | tokens {step*tokens_per_step/1e6:6.1f}M | {(time.time()-t0)/3600:.2f} h | "
                f"scale {scaler.get_scale():.0f}")

        if not args.bench and step % args.eval_every == 0:
            if master:
                r = evaluate()
                history["val"].append({"step": step, "tokens": step * tokens_per_step, **r})
                maybe_save_best(step, r)
                tqdm.write("   VAL " + " | ".join(f"{k} {v:.4f}" for k, v in r.items()))
            if use_ddp:
                dist.barrier()

        if not args.bench and step % args.ckpt_every == 0:
            if master:
                save_ckpt(step, time.time() - t0)
            if use_ddp:
                dist.barrier()

    pbar.close()

    if not args.bench:
        if master:
            r = evaluate()
            history["val"].append({"step": step, "tokens": step * tokens_per_step, **r})
            maybe_save_best(step, r)
            tqdm.write("FINAL VAL " + " | ".join(f"{k} {v:.4f}" for k, v in r.items()))
            save_ckpt(step, time.time() - t0)
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg)}, os.path.join(args.out_dir, "final_model.pt"))
            tqdm.write(f"saved final_model.pt after {step} steps ({step*tokens_per_step/1e6:.0f}M tokens, "
               f"{(time.time()-t0)/3600:.2f} h)")
        if use_ddp:
            dist.barrier()
    if use_ddp:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
'''!PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node=2 train_llm.py \
  --preset tiny --attn_impl sdpa --seq_len 128 --global_bs 8 \
  --total_tokens 100e5 --time_hours 0.3 --warmup_steps 10 \
  --log_every 10 --eval_every 50 --ckpt_every 100 --eval_batches 2 \
  --out_dir /kaggle/working/smoke_ckpt'''

In [ ]:
'''# =====================================================================
# INFERENCE & VERIFICATION CELL (Smoke Test & Full Model compatible)
# =====================================================================
import os
import json
import zipfile
import torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from llama_scratch import Config, LlamaLM, generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Locate Checkpoint (checks smoke_ckpt first, then full training checkpoint)
ckpt_path = "/kaggle/working/smoke_ckpt/final_model.pt"
if not os.path.exists(ckpt_path):
    ckpt_path = "/kaggle/working/cs781_phase1/model_checkpoint/final_model.pt"

print(f"Loading weights from: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cpu")
cfg = Config(**ckpt["cfg"])
model = LlamaLM(cfg).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

# Load Tokenizer
tok = PureByteLevelBPETokenizer.load(TOKENIZER_DIR)

# ---------------------------------------------------------------------
# A. Quick Visual Generation Check (See what the model learned!)
# ---------------------------------------------------------------------
test_prompt = "PETITIONER: THE STATE OF MAHARASHTRA Vs. RESPONDENT: UNION OF INDIA"
print("\n" + "="*50)
print(f"PROMPT: {test_prompt}")

input_ids = tok.encode(test_prompt)
out_ids = generate(
    model=model,
    prompts=[input_ids],
    max_new_tokens=64,
    eos_id=tok.eos_token_id,
    pad_id=tok.pad_token_id,
    temperature=0.3,
    top_p=0.9,
    rep_penalty=1.2
)

continuation = tok.decode(out_ids[0])
print(f"\nMODEL CONTINUATION:\n{continuation}")
print("="*50 + "\n")

# ---------------------------------------------------------------------
# B. Generate Official Competition Predictions & submission.zip
# ---------------------------------------------------------------------
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Generating continuations for {len(test_data)} test prompts...")

predictions = {}
items = list(test_data.items())
BATCH_SIZE = 16

for start in tqdm(range(0, len(items), BATCH_SIZE), desc="Generating Test Continuations"):
    batch = items[start:start + BATCH_SIZE]
    batch_ids = [str(k) for k, _ in batch]
    batch_prompts = [rec["text"] for _, rec in batch]
    
    # Encode and truncate to context window
    encoded_prompts = [tok.encode(t)[-(cfg.max_seq_len - 256):] for t in batch_prompts]
    
    gen_batches = generate(
        model=model,
        prompts=encoded_prompts,
        max_new_tokens=256,
        eos_id=tok.eos_token_id,
        pad_id=tok.pad_token_id,
        temperature=0.2,
        top_p=0.95,
        rep_penalty=1.15
    )
    
    for ex_id, g_ids in zip(batch_ids, gen_batches):
        text = tok.decode(g_ids)
        # Competition requirement 7.2.5: Exactly 128 words, continuation only
        words = text.split()[:128]
        predictions[ex_id] = {"text": " ".join(words)}

# Save prediction.json
pred_path = "/kaggle/working/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

# Create submission.zip (containing prediction.json at root)
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

print(f"Generated {pred_path} and {zip_path} successfully!")'''

In [ ]:
import os, sys, json, time, shutil, zipfile, importlib
import numpy as np, torch
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo, hf_hub_download

MODE   = "full"            # "smoke2" or "full"
PRESET = "target_150m"       # same in both modes (142.0M params). "base" = 201.6M
RUNS = {
 "smoke2": dict(ckpt_dir="/kaggle/working/smoke2_ckpt", out_dir="/kaggle/working/smoke2_out",
                tokens=8_000_000,   time_h=0.4, warmup=10,  n_eval=96,  suffix="-smoke2"),
 "full":   dict(ckpt_dir="/kaggle/working/cs781_phase1/model_checkpoint", out_dir="/kaggle/working/final_out",
                tokens=200_000_000, time_h=4.25, warmup=100, n_eval=300, suffix=""),
}
R = RUNS[MODE]
HF_MODEL_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-llm-phase1{R['suffix']}"
os.makedirs(R["out_dir"], exist_ok=True); sys.path.append("/kaggle/working")
print(MODE, "->", HF_MODEL_REPO)

In [ ]:
TOK_, TIME_, WARM_, PRE_, CK_ = R["tokens"], R["time_h"], R["warmup"], PRESET, R["ckpt_dir"]
RESUME = ""        # set to "--resume" after a crash; for smoke2 also test: tokens=10_000_000 + "--resume"
EB = 8 if MODE == "smoke2" else 32
!PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node=2 train_llm.py --preset {PRE_} --attn_impl sdpa --seq_len 1024 --global_bs 128 --total_tokens {TOK_} --time_hours {TIME_} --warmup_steps {WARM_} --eval_batches {EB} --out_dir {CK_} {RESUME} 2>&1 | tee /kaggle/working/train_{MODE}.log | grep --line-buffered -E "GPUs|total |seq_len|unique|VAL|best|saved|resumed|Error|Traceback|step +[0-9]+/"

In [ ]:
_api = HfApi(token=HF_TOKEN); create_repo(HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True, token=HF_TOKEN)
_api.upload_file(path_or_fileobj=f"{CK_}/final_model.pt", path_in_repo="native/final_model.pt", repo_id=HF_MODEL_REPO, repo_type="model")

In [ ]:
!pip install -q bert-score rouge-score sacrebleu
import llama_scratch; importlib.reload(llama_scratch)
from llama_scratch import Config, LlamaLM, generate, count_params
tok, MAX_NEW = tokenizer, 288

def load_model(path):
    ck = torch.load(path, map_location="cpu")
    m = LlamaLM(Config(**ck["cfg"])); m.load_state_dict(ck["model"])
    return m.to(device).eval(), ck

def make_heldout(n, seed=0, domains=("supreme_court", "high_courts", "tribunals", "acts")):
    rng, out, per = np.random.default_rng(seed), [], max(1, n // len(domains))
    for d in domains:
        a = np.memmap(f"{TOKEN_DATA_DIR}/val_{d}.bin", dtype=np.uint16, mode="r")
        ends = np.flatnonzero(a == tok.eos_token_id); starts = np.r_[0, ends[:-1] + 1]; got = 0
        for i in rng.permutation(len(starts)):
            if ends[i] - starts[i] < 1200: continue
            w = tok.decode(a[starts[i]:ends[i]].tolist()).split()
            if len(w) < 700: continue
            s = int(rng.integers(0, len(w) - 640))
            out.append(dict(domain=d, prompt=" ".join(w[s:s+512]), ref=" ".join(w[s+512:s+640])))
            got += 1
            if got == per: break
    return out

def metrics(cands, refs):
    import bert_score, sacrebleu
    from rouge_score import rouge_scorer
    rs = rouge_scorer.RougeScorer(["rougeL"])
    rl = float(np.mean([rs.score(r, c)["rougeL"].fmeasure for c, r in zip(cands, refs)]))
    bl = sacrebleu.corpus_bleu(cands, [refs]).score / 100
    _, _, F = bert_score.score(cands, refs, lang="en", batch_size=16, device=str(device))
    return dict(BERTScore_F1=float(F.mean()), ROUGE_L_F1=rl, BLEU4=bl)

@torch.no_grad()
def run_generation(model, texts, bs=32, **dec):
    maxp = model.cfg.max_seq_len - MAX_NEW
    assert maxp > 256, "max_seq_len too small"
    outs = []
    for i in tqdm(range(0, len(texts), bs), desc="generate", mininterval=30):
        batch = [tok.encode(t)[-maxp:] for t in texts[i:i+bs]]
        g = generate(model, batch, MAX_NEW, tok.eos_token_id, tok.pad_token_id, **dec)
        #outs += [" ".join(tok.decode(x).split()[:128]) for x in g]
        outs += [" ".join(tok.decode(x).split()[:128]) or "." for x in g]
    return outs

show = lambda d: {k: round(v, 4) for k, v in d.items()}
CANDS = {n: f"{R['ckpt_dir']}/{n}" for n in ("best_model.pt", "final_model.pt") if os.path.exists(f"{R['ckpt_dir']}/{n}")}
held = make_heldout(R["n_eval"]); P = [h["prompt"] for h in held]; REF = [h["ref"] for h in held]
print(len(held), "held-out prompts")

scores = {}
for name, path in CANDS.items():
    m, ck = load_model(path)
    scores[name] = metrics(run_generation(m, P, temperature=0.2, top_p=0.95, rep_penalty=1.0), REF)
    print(name, "step", ck.get("step"), show(scores[name])); del m; torch.cuda.empty_cache()
CHOSEN = max(scores, key=lambda n: scores[n]["BERTScore_F1"])

print("\ncontrols:")
for k, v in {"copy last 128 prompt words": [" ".join(p.split()[-128:]) for p in P],
             "other doc's continuation":  REF[1:] + REF[:1]}.items():
    print(f"  {k:28s}", show(metrics(v, REF)))

GRID = {"greedy": dict(temperature=0.0, rep_penalty=1.0),
        "T0.2 p0.95": dict(temperature=0.2, top_p=0.95, rep_penalty=1.0),
        "T0.2 p0.95 rp1.1": dict(temperature=0.2, top_p=0.95, rep_penalty=1.1),
        "T0.5 p0.9": dict(temperature=0.5, top_p=0.9, rep_penalty=1.0)}
m, _ = load_model(CANDS[CHOSEN]); res = {}
for k, dec in GRID.items():
    res[k] = metrics(run_generation(m, P, **dec), REF); print(f"  {k:18s}", show(res[k]))
DEC_NAME = max(res, key=lambda k: res[k]["BERTScore_F1"]); DEC = GRID[DEC_NAME]
print("chosen:", CHOSEN, "+", DEC_NAME); del m; torch.cuda.empty_cache()

In [ ]:
test_path = globals().get("test_path") or hf_hub_download("Exploration-Lab/CS-781-Capstone",
            "Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=HF_TOKEN)
model, ck = load_model(CANDS[CHOSEN])
test = json.load(open(test_path, encoding="utf-8")); tids = list(test); ttexts = [test[i]["text"] for i in tids]
tl = np.array([len(tok.encode(t)) for t in ttexts])
print(f"{len(tids)} prompts | tokens p50 {np.percentile(tl,50):.0f} p95 {np.percentile(tl,95):.0f} max {tl.max()} | prompt budget {model.cfg.max_seq_len - MAX_NEW}")

t0 = time.time(); outs = run_generation(model, ttexts, **DEC); print(f"generation: {(time.time()-t0)/60:.1f} min")
pred = {i: {"text": o} for i, o in zip(tids, outs)}
PRED, ZIP = f"{R['out_dir']}/prediction.json", f"{R['out_dir']}/submission.zip"
json.dump(pred, open(PRED, "w", encoding="utf-8"), ensure_ascii=False)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED, arcname="prediction.json")

assert set(pred) == set(test), "ID mismatch"
assert all(v["text"].strip() for v in pred.values()), "empty prediction"
assert zipfile.ZipFile(ZIP).namelist() == ["prediction.json"]
wc = np.array([len(v["text"].split()) for v in pred.values()])
same = sum(pred[i]["text"].split()[:8] == test[i]["text"].split()[:8] for i in pred)
print(f"words/pred min {wc.min()} median {np.median(wc):.0f} | <128: {(wc<128).sum()} | begins like its prompt: {same}\nOK -> {ZIP}")

In [ ]:
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=ZIP, path_in_repo="submission.zip", repo_id=HF_MODEL_REPO, repo_type="model")
print(f"https://huggingface.co/{HF_MODEL_REPO}/blob/main/submission.zip")

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, AddedToken
from transformers import (LlamaConfig, LlamaForCausalLM, AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast)
ALLOW_OVERWRITE = False
PKG = f"{R['out_dir']}/hf_package"; shutil.rmtree(PKG, ignore_errors=True); os.makedirs(PKG)
cfg = model.cfg
assert cfg.pos_enc == "rope" and cfg.rope_theta == 10000.0 and cfg.tie_embeddings

# tokenizer -> standard tokenizer.json (same ids as your class on legal text; no custom code needed)
bs_ = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs_ = bs_[:]; n_ = 0
for b in range(256):
    if b not in bs_: bs_.append(b); cs_.append(256 + n_); n_ += 1
b2u = dict(zip(bs_, map(chr, cs_))); enc = lambda bb: "".join(b2u[x] for x in bb)
spec = {s.encode() for s in tok.special_tokens}
vocab = {(bb.decode() if bb in spec else enc(bb)): i for bb, i in tok.encoder.items()}
tk = Tokenizer(models.BPE(vocab=vocab, merges=[(enc(a), enc(b)) for a, b in tok.merges]))
tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True); tk.decoder = decoders.ByteLevel()
tk.add_special_tokens([AddedToken(s, special=True) for s in tok.special_tokens])
PreTrainedTokenizerFast(tokenizer_object=tk, bos_token="<s>", eos_token="</s>", pad_token="<pad>",
                        unk_token="<unk>", model_max_length=cfg.max_seq_len).save_pretrained(PKG)

# model -> transformers LlamaForCausalLM layout (same math: RMSNorm, RoPE half-split, GQA, SwiGLU)
hcfg = LlamaConfig(vocab_size=cfg.vocab_size, hidden_size=cfg.d_model, intermediate_size=cfg.ffn_hidden,
    num_hidden_layers=cfg.n_layers, num_attention_heads=cfg.n_heads, num_key_value_heads=cfg.n_kv_heads,
    max_position_embeddings=cfg.max_seq_len, rms_norm_eps=cfg.norm_eps, hidden_act="silu",
    attention_bias=False, mlp_bias=False, tie_word_embeddings=True,
    bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
sd = {k: v.float().cpu() for k, v in model.state_dict().items()}
new = {"model.embed_tokens.weight": sd["tok_emb.weight"], "lm_head.weight": sd["tok_emb.weight"],
       "model.norm.weight": sd["norm.weight"]}
for i in range(cfg.n_layers):
    p, q = f"layers.{i}.", f"model.layers.{i}."
    new[q+"input_layernorm.weight"] = sd[p+"attn_norm.weight"]
    new[q+"post_attention_layernorm.weight"] = sd[p+"mlp_norm.weight"]
    for a, b in (("wq","q_proj"), ("wk","k_proj"), ("wv","v_proj"), ("wo","o_proj")):
        new[q+f"self_attn.{b}.weight"] = sd[p+f"attn.{a}.weight"]
    for a, b in (("w_gate","gate_proj"), ("w_up","up_proj"), ("w_down","down_proj")):
        new[q+f"mlp.{b}.weight"] = sd[p+f"mlp.{a}.weight"]
hf = LlamaForCausalLM(hcfg)
missing, unexpected = hf.load_state_dict(new, strict=False)
assert not unexpected and all(k == "lm_head.weight" for k in missing), (missing, unexpected)
hf.tie_weights(); hf.save_pretrained(PKG, safe_serialization=True)

# provenance files (in a subfolder so they can't clash with HF's own files)
os.makedirs(f"{PKG}/native/custom_bpe", exist_ok=True)
shutil.copy("/kaggle/working/llama_scratch.py", f"{PKG}/native/llama_scratch.py")
for f in os.listdir(TOKENIZER_DIR): shutil.copy(f"{TOKENIZER_DIR}/{f}", f"{PKG}/native/custom_bpe/{f}")
if os.path.exists(f"{R['ckpt_dir']}/history.json"): shutil.copy(f"{R['ckpt_dir']}/history.json", f"{PKG}/native/history.json")
open(f"{PKG}/README.md", "w").write(
  f"# {HF_MODEL_REPO}\nCS-781 Phase 1 legal LM, trained from scratch.\n- {count_params(model, verbose=False)/1e6:.1f}M params (tied embeddings)\n"
  f"- Llama-style: RMSNorm, RoPE, GQA {cfg.n_heads}/{cfg.n_kv_heads}, SwiGLU; context {cfg.max_seq_len}\n"
  f"- byte-level BPE, 33k vocab, trained from scratch\n- checkpoint: {os.path.basename(CANDS[CHOSEN])}, step {ck.get('step','final')}\n"
  f"- load: AutoModelForCausalLM.from_pretrained(repo), AutoTokenizer.from_pretrained(repo)\n")

# local check: reload from disk, compare logits with the native model
chk = AutoModelForCausalLM.from_pretrained(PKG).float().to(device).eval()
x = torch.tensor([tok.encode(ttexts[0])[:600]], device=device)
with torch.no_grad(): a = model.float()(x, all_logits=True)[0].float(); d1 = (a - chk(x).logits.float()).abs().max().item()
print("max |logit diff| native vs exported:", d1); assert d1 < 1e-3
assert AutoTokenizer.from_pretrained(PKG)(ttexts[0], add_special_tokens=False)["input_ids"] == tok.encode(ttexts[0])

# upload (refuses to overwrite an existing model unless you say so), then verify from the Hub
api = HfApi(token=HF_TOKEN)
create_repo(HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True, token=HF_TOKEN)
if api.file_exists(HF_MODEL_REPO, "model.safetensors", repo_type="model") and not ALLOW_OVERWRITE:
    raise RuntimeError(f"{HF_MODEL_REPO} already has a model: use another repo or set ALLOW_OVERWRITE=True")
info = api.upload_folder(folder_path=PKG, repo_id=HF_MODEL_REPO, repo_type="model",
                         commit_message=f"{MODE}: {os.path.basename(CANDS[CHOSEN])} ({PRESET})")
print("uploaded:", info.commit_url)
back = AutoModelForCausalLM.from_pretrained(HF_MODEL_REPO, token=HF_TOKEN).float().to(device).eval()
with torch.no_grad(): d2 = (a - back(x).logits.float()).abs().max().item()
print("max |logit diff| native vs re-downloaded from the Hub:", d2); assert d2 < 1e-3
if MODE == "full": api.create_tag(HF_MODEL_REPO, tag="phase1-final", repo_type="model")

In [ ]:
'''HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=ZIP, path_in_repo="submission.zip", repo_id=HF_MODEL_REPO, repo_type="model")
print(f"https://huggingface.co/{HF_MODEL_REPO}/blob/main/submission.zip")'''

In [ ]:
# check if currently I'm choosing the best checkpoint
# I'm outputting too much
# end to end run
# what ultimate val loss should I be able to acheive
# how did it train on 10 million tokens so far, please also calculate its bleu score or wtv